In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import random

In [ ]:
import requests
from bs4 import BeautifulSoup
import time
import pandas as pd

catalog_url = "https://www.jumia.com.ng/mlp-official-stores/?page={page}catalog-listing"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9"
}

ml_dataset = []

for page in range(1, 26):
    print(f"\nScraping Catalog Page {page}...")
    url = catalog_url.format(page=page)
    
    try:
        response = requests.get(url, headers=headers)
        if response.status_code != 200:
            print(f"Skipping page {page} - Status Code: {response.status_code}")
            continue
            
        soup = BeautifulSoup(response.content, "html.parser")
        items = soup.find_all("article", class_="prd _fb col c-prd")

        if not items:
            print(f"No items found on page {page}.")
            break

        for item in items:
            title_tag = item.find("h3", class_="name")
            current_price_tag = item.find("div", class_="prc")
            old_price_tag = item.find("div", class_="old")
            
            img_tag = item.find("img", class_="img")
            image_link = img_tag.get("data-src") or img_tag.get("src") if img_tag else "No Image"
            
            title = title_tag.text.strip() if title_tag else "No Title"
            current_price = current_price_tag.text.strip() if current_price_tag else "No Price"
            old_price = old_price_tag.text.strip() if old_price_tag else "No Old Price"
            
            link_tag = item.find("a", class_="core")
            product_link = "https://www.jumia.com.ng" + link_tag['href'] if link_tag and 'href' in link_tag.attrs else None
            
            if product_link:
                try:
                    pdp_response = requests.get(product_link, headers=headers)
                    
                    if pdp_response.status_code == 200:
                        pdp_soup = BeautifulSoup(pdp_response.content, "html.parser")
                        
                        breadcrumbs = pdp_soup.find_all("a", class_="cbs")
                        category = " > ".join([cb.text.strip() for cb in breadcrumbs]) if breadcrumbs else "No Category"
                        
                        rating_count_tag = pdp_soup.find("a", class_="-plxs _more")
                        total_ratings = rating_count_tag.text.strip() if rating_count_tag else "0 ratings"
                        
                        discount_tag = pdp_soup.find("span", class_="bdg _dsct _dyn -mls")
                        discount = discount_tag.text.strip() if discount_tag else "0%"
                        
                        reviews = pdp_soup.find_all("article", class_="-pvs -hr _bet")
                        
                        if not reviews:
                            ml_dataset.append([title, category, current_price, old_price, discount, total_ratings, image_link, "No Rating", "No Date", "No", "No comments"])
                        else:
                            for rev in reviews:
                                rating_tag = rev.find("div", class_="stars")
                                comment_tag = rev.find("p", class_="-pvs")
                                
                                date_text = "No Date"
                                verified_status = "No"
                                
                                if rev.find(string=lambda text: text and "Verified Purchase" in text):
                                    verified_status = "Yes"
                                    
                                spans = rev.find_all("span")
                                for span in spans:
                                    span_text = span.text.strip()
                                    if " by " in span_text and "-" in span_text:
                                        date_text = span_text.split(" by ")[0].strip()
                                        break
                                
                                rating_text = rating_tag.text.strip() if rating_tag else "No Rating"
                                comment_text = comment_tag.text.strip() if comment_tag else "No comments"
                                
                                ml_dataset.append([
                                    title, 
                                    category,
                                    current_price, 
                                    old_price, 
                                    discount,
                                    total_ratings,
                                    image_link,
                                    rating_text,
                                    date_text,           
                                    verified_status,     
                                    comment_text
                                ])
                    
                    time.sleep(1.5) 
                    
                except requests.exceptions.RequestException as e:
                    print(f"  Error fetching product page: {e}")
                    
    except requests.exceptions.RequestException as e:
         print(f"Catalog page error: {e}")
         
    time.sleep(2)


print("\nScraping complete! Formatting data for Machine Learning...")

columns = [
    "Product_Name", "Category", "Current_Price", "Old_Price", 
    "Discount", "Total_Ratings", "Image_Link", 
    "Review_Rating", "Review_Date", "Verified_Purchase", "Review_Text"
]
df = pd.DataFrame(ml_dataset, columns=columns)

filename = "jumia_dataset.csv"
df.to_csv(filename, index=False)

print(f"Successfully saved {len(df)} review samples to {filename}!")
print("\nDataset Preview:")
display(df.head())


Scraping Catalog Page 1...

Scraping Catalog Page 2...
  Error fetching product page: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))

Scraping Catalog Page 3...

Scraping Catalog Page 4...
  Error fetching product page: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))

Scraping Catalog Page 5...
  Error fetching product page: HTTPSConnectionPool(host='www.jumia.com.ng', port=443): Read timed out. (read timeout=None)

Scraping Catalog Page 6...
Catalog page error: HTTPSConnectionPool(host='www.jumia.com.ng', port=443): Max retries exceeded with url: /mlp-official-stores/?page=6catalog-listing (Caused by SSLError(SSLError(1, '[SSL: DECRYPTION_FAILED_OR_BAD_RECORD_MAC] decryption failed or bad record mac (_ssl.c:1006)')))

Scraping Catalog Page 7...

Scraping Catalog Page 8...

Scraping Catalog Page 9...

Scraping Catalog Page 10...

Scraping Catalog Page 11...

Scraping Catalog Page 12...

Scraping

,Product_Name,Category,Current_Price,Old_Price,Discount,Total_Ratings,Image_Link,Review_Rating,Review_Date,Verified_Purchase,Review_Text
0,Philly 20000mAh Power Charging Bank Type-C Input,Home > Phones & Tablets > Mobile Phone Accesso...,"₦ 8,490","₦ 13,875",39%,(12459 verified ratings),https://ng.jumia.is/unsafe/fit-in/300x300/filt...,5 out of 5,No Date,Yes,Portable in size
1,Philly 20000mAh Power Charging Bank Type-C Input,Home > Phones & Tablets > Mobile Phone Accesso...,"₦ 8,490","₦ 13,875",39%,(12459 verified ratings),https://ng.jumia.is/unsafe/fit-in/300x300/filt...,5 out of 5,No Date,Yes,Very durable
2,Philly 20000mAh Power Charging Bank Type-C Input,Home > Phones & Tablets > Mobile Phone Accesso...,"₦ 8,490","₦ 13,875",39%,(12459 verified ratings),https://ng.jumia.is/unsafe/fit-in/300x300/filt...,5 out of 5,No Date,Yes,It has the quality as expected
3,Philly 20000mAh Power Charging Bank Portable C...,Home > Phones & Tablets > Mobile Phone Accesso...,"₦ 7,752","₦ 13,057",41%,(18675 verified ratings),https://ng.jumia.is/unsafe/fit-in/300x300/filt...,1 out of 5,No Date,Yes,Strong and powerful. Been using for a year
4,Philly 20000mAh Power Charging Bank Portable C...,Home > Phones & Tablets > Mobile Phone Accesso...,"₦ 7,752","₦ 13,057",41%,(18675 verified ratings),https://ng.jumia.is/unsafe/fit-in/300x300/filt...,5 out of 5,No Date,Yes,Always on full bar


In [8]:
df

,Product_Name,Category,Current_Price,Old_Price,Discount,Total_Ratings,Image_Link,Review_Rating,Review_Date,Verified_Purchase,Review_Text
0,Philly 20000mAh Power Charging Bank Type-C Input,Home > Phones & Tablets > Mobile Phone Accesso...,"₦ 8,490","₦ 13,875",39%,(12459 verified ratings),https://ng.jumia.is/unsafe/fit-in/300x300/filt...,5 out of 5,No Date,Yes,Portable in size
1,Philly 20000mAh Power Charging Bank Type-C Input,Home > Phones & Tablets > Mobile Phone Accesso...,"₦ 8,490","₦ 13,875",39%,(12459 verified ratings),https://ng.jumia.is/unsafe/fit-in/300x300/filt...,5 out of 5,No Date,Yes,Very durable
2,Philly 20000mAh Power Charging Bank Type-C Input,Home > Phones & Tablets > Mobile Phone Accesso...,"₦ 8,490","₦ 13,875",39%,(12459 verified ratings),https://ng.jumia.is/unsafe/fit-in/300x300/filt...,5 out of 5,No Date,Yes,It has the quality as expected
3,Philly 20000mAh Power Charging Bank Portable C...,Home > Phones & Tablets > Mobile Phone Accesso...,"₦ 7,752","₦ 13,057",41%,(18675 verified ratings),https://ng.jumia.is/unsafe/fit-in/300x300/filt...,1 out of 5,No Date,Yes,Strong and powerful. Been using for a year
4,Philly 20000mAh Power Charging Bank Portable C...,Home > Phones & Tablets > Mobile Phone Accesso...,"₦ 7,752","₦ 13,057",41%,(18675 verified ratings),https://ng.jumia.is/unsafe/fit-in/300x300/filt...,5 out of 5,No Date,Yes,Always on full bar
...,...,...,...,...,...,...,...,...,...,...,...
2707,Oraimo PowerTrans USB-C Hub 7-in-1 Multi-Funct...,Home > Computing > Computers > Networking Prod...,"₦ 24,000","₦ 37,500",36%,(13 verified ratings),https://ng.jumia.is/unsafe/fit-in/300x300/filt...,4 out of 5,No Date,Yes,As advertised
2708,Danami Men's Plain Round Neck T-Shirt- White,Home > Fashion > Men's Fashion > Clothing > Sh...,"₦ 6,900 - ₦ 7,900",No Old Price,0%,(587 verified ratings),https://ng.jumia.is/unsafe/fit-in/300x300/filt...,4 out of 5,No Date,Yes,Good enough
2709,Danami Men's Plain Round Neck T-Shirt- White,Home > Fashion > Men's Fashion > Clothing > Sh...,"₦ 6,900 - ₦ 7,900",No Old Price,0%,(587 verified ratings),https://ng.jumia.is/unsafe/fit-in/300x300/filt...,5 out of 5,No Date,Yes,Perfect
2710,Danami Men's Plain Round Neck T-Shirt- White,Home > Fashion > Men's Fashion > Clothing > Sh...,"₦ 6,900 - ₦ 7,900",No Old Price,0%,(587 verified ratings),https://ng.jumia.is/unsafe/fit-in/300x300/filt...,5 out of 5,No Date,Yes,The top isn't bad
